<a href="https://colab.research.google.com/github/Niharika9948/NLP_Assignments/blob/main/Assignment_12_2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [4]:
# ===============================
# STEP 2 — Import Libraries
# ===============================
import numpy as np
import pandas as pd
import re
from collections import Counter
import urllib.request

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset

from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix


# ===============================
# STEP 3 — Download & Load Dataset
# ===============================
# Automatically download dataset
url = "https://raw.githubusercontent.com/justmarkham/pycon-2016-tutorial/master/data/sms.tsv"
urllib.request.urlretrieve(url, "SMSSpamCollection")

# Load dataset
df = pd.read_csv("SMSSpamCollection", sep='\t', names=['label', 'text'])

print("Sample Data:\n", df.head())
print("\nClass Distribution:\n", df['label'].value_counts())


# ===============================
# STEP 4 — Text Preprocessing
# ===============================
def preprocess(text):
    text = text.lower()  # lowercase
    text = re.sub(r'[^\w\s]', '', text)  # remove punctuation
    tokens = text.split()  # tokenization
    return tokens

df['tokens'] = df['text'].apply(preprocess)

# Padding sequences
MAX_LEN = 50

def pad_sequence(seq, max_len):
    if len(seq) < max_len:
        seq += ['<PAD>'] * (max_len - len(seq))
    else:
        seq = seq[:max_len]
    return seq

df['tokens'] = df['tokens'].apply(lambda x: pad_sequence(x, MAX_LEN))


# ===============================
# STEP 5 — Vocabulary & Encoding
# ===============================
all_tokens = [token for tokens in df['tokens'] for token in tokens]
vocab = Counter(all_tokens)

word2idx = {word: idx+1 for idx, (word, _) in enumerate(vocab.items())}
word2idx['<PAD>'] = 0

def encode(tokens):
    return [word2idx[word] for word in tokens]

df['encoded'] = df['tokens'].apply(encode)

# Encode labels
df['label'] = df['label'].map({'ham': 0, 'spam': 1})


# ===============================
# STEP 6 — Train-Test Split
# ===============================
X = np.array(df['encoded'].tolist())
y = np.array(df['label'].tolist())

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)


# ===============================
# STEP 7 — Multi-Filter CNN Model
# ===============================
class TextCNN(nn.Module):
    def __init__(self, vocab_size, embed_dim, num_classes):
        super(TextCNN, self).__init__()

        self.embedding = nn.Embedding(vocab_size, embed_dim)

        # Multiple convolution filters
        self.conv1 = nn.Conv1d(embed_dim, 100, kernel_size=3)
        self.conv2 = nn.Conv1d(embed_dim, 100, kernel_size=4)
        self.conv3 = nn.Conv1d(embed_dim, 100, kernel_size=5)

        self.dropout = nn.Dropout(0.5)
        self.fc = nn.Linear(300, num_classes)

    def forward(self, x):
        x = self.embedding(x)          # (batch, seq_len, embed_dim)
        x = x.permute(0, 2, 1)         # (batch, embed_dim, seq_len)

        x1 = torch.relu(self.conv1(x))
        x2 = torch.relu(self.conv2(x))
        x3 = torch.relu(self.conv3(x))

        # Global Max Pooling
        x1 = torch.max(x1, dim=2)[0]
        x2 = torch.max(x2, dim=2)[0]
        x3 = torch.max(x3, dim=2)[0]

        # Concatenate feature maps
        x = torch.cat((x1, x2, x3), dim=1)

        x = self.dropout(x)
        out = self.fc(x)

        return out


# ===============================
# STEP 8 — Model Training
# ===============================
X_train_tensor = torch.tensor(X_train, dtype=torch.long)
y_train_tensor = torch.tensor(y_train, dtype=torch.long)

train_dataset = TensorDataset(X_train_tensor, y_train_tensor)
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)

# Fix: Increase vocab_size by 1 to correctly include all indices
model = TextCNN(vocab_size=len(word2idx) + 1, embed_dim=100, num_classes=2)

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

EPOCHS = 5

for epoch in range(EPOCHS):
    total_loss = 0

    for inputs, labels in train_loader:
        optimizer.zero_grad()

        outputs = model(inputs)
        loss = criterion(outputs, labels)

        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    print(f"Epoch {epoch+1}/{EPOCHS}, Loss: {total_loss:.4f}")


# ===============================
# STEP 9 — Model Evaluation
# ===============================
X_test_tensor = torch.tensor(X_test, dtype=torch.long)

model.eval()
with torch.no_grad():
    outputs = model(X_test_tensor)
    preds = torch.argmax(outputs, dim=1).numpy()

print("\nEvaluation Metrics:")
print("Accuracy:", accuracy_score(y_test, preds))
print("Precision:", precision_score(y_test, preds))
print("Recall:", recall_score(y_test, preds))
print("F1 Score:", f1_score(y_test, preds))
print("Confusion Matrix:\n", confusion_matrix(y_test, preds))

Sample Data:
   label                                               text
0   ham  Go until jurong point, crazy.. Available only ...
1   ham                      Ok lar... Joking wif u oni...
2  spam  Free entry in 2 a wkly comp to win FA Cup fina...
3   ham  U dun say so early hor... U c already then say...
4   ham  Nah I don't think he goes to usf, he lives aro...

Class Distribution:
 label
ham     4825
spam     747
Name: count, dtype: int64
Epoch 1/5, Loss: 33.7079
Epoch 2/5, Loss: 12.3081
Epoch 3/5, Loss: 6.2310
Epoch 4/5, Loss: 3.3107
Epoch 5/5, Loss: 2.3044

Evaluation Metrics:
Accuracy: 0.9820627802690582
Precision: 0.9708029197080292
Recall: 0.8926174496644296
F1 Score: 0.9300699300699301
Confusion Matrix:
 [[962   4]
 [ 16 133]]
